# **TENTATIVE de création de RESEAU DE NEURONES pour le bpm**

Import des fichiers et Conexion au drive



In [ ]:

import os
import glob
import numpy as np
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
import librosa
import librosa.display

# TensorFlow / Keras
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Flatten, Dropout, Conv2D, MaxPooling2D, BatchNormalization

# Pour connecter le Drive
from google.colab import drive

# --- CONSTANTES POUR LA MUSIQUE (DIFFÉRENT DES CHIFFRES) ---
# Pour la parole, 8000Hz suffisait. Pour la musique, le rythme est souvent dans les basses et les percussions aiguës 22050Hz est un standard
SAMPLE_RATE = 22050

# Durée du morceau qu'on va analyser (en secondes).
# Contrairement aux chiffres prononcé par les speakers durant le PJT1A (les fichiers avaient une durée de 1se), pour le BPM il faut écouter plus longtemps.
# On coupe tous les morceaux à  secondes pour avoir une entrée fixe pour le réseau de neurone .
DURATION = 30
SAMPLES_PER_TRACK = SAMPLE_RATE * DURATION

def connexion_drive(working_dir):
  print("Connexion au drive...")
  drive.mount('/content/drive', force_remount=True)
  if not os.path.exists(working_dir):
    os.mkdir(working_dir)
  print(f"Connecté ! Dossier de travail : {working_dir}")

# --- CONFIGURATION DES CHEMINS (À MODIFIER SI BESOIN) ---
working_dir = 'drive/MyDrive/PJT IA Musicale'

data_dir=working_dir+"/Datasets"

# Utilisation des fonctions d'initialisation
connexion_drive(working_dir)


data_dir=working_dir+"/Datasets"+"/converted_wav" #permet d'obtenir le chemin relatif de tous les fichiers du dossier recordings

# Populate list_of_files with the converted WAV files
import glob
list_of_files = glob.glob(data_dir + "/*.wav")

i=0
print(list_of_files[0])

Connexion au drive...
Mounted at /content/drive
Connecté ! Dossier de travail : drive/MyDrive/PJT IA Musicale
drive/MyDrive/PJT IA Musicale/Datasets/converted_wav/93 -  - When You Say My Name.wav


création du fichier sur notre drive pour l'importation des des nouvelles données


In [ ]:
from google.colab import drive
import os

drive.mount('/content/drive')

# Créer le dossier de destination s'il n'existe pas
destination_folder = '/content/drive/MyDrive/PJT IA Musicale/Datasets/GiantSteps'
if not os.path.exists(destination_folder):
    os.makedirs(destination_folder)
    print(f"Dossier créé : {destination_folder}")
else:
    print(f"Le dossier existe déjà : {destination_folder}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Le dossier existe déjà : /content/drive/MyDrive/PJT IA Musicale/Datasets/GiantSteps


On importe depuis github les fichiers audio et on les stocke localement sur google collab


On les a trouvé sur cette page précisemment: https://github.com/GiantSteps/giantsteps-tempo-dataset.git

In [ ]:
# 1. On se place dans l'espace temporaire de Colab
%cd /content/

# 2. On clone le dépôt GitHub (récupérer le code)
!git clone https://github.com/GiantSteps/giantsteps-tempo-dataset.git

# 3. On entre dans le dossier du dépôt
%cd giantsteps-tempo-dataset

# 4. On donne la permission d'exécuter le script de téléchargement
!chmod +x audio_dl.sh

# 5. On lance le téléchargement (Cela va prendre quelques minutes pour 664 fichiers)
print("Téléchargement des fichiers audio en cours...")
!./audio_dl.sh

print("Téléchargement terminé !")

61087.LOFI.mp3 ...  
  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100 1406k  100 1406k    0     0   919k      0  0:00:01  0:00:01 --:--:--  920k
MD5 OK! 

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100 1406k  100 1406k    0     0   916k      0  0:00:01  0:00:01 --:--:--  915k
^C
Téléchargement terminé !


On importe désormais tous les fichiers audio vers le drive


In [ ]:
import shutil
import os

# Dossier où le script a téléchargé les sons
source_audio = '/content/giantsteps-tempo-dataset/audio'

target_audio = '/content/drive/MyDrive/PJT IA Musicale/Datasets/GiantSteps'

print(f"Copie des fichiers de {source_audio} vers {target_audio}")

# On déplace les fichiers (shutil.copytree serait plus direct mais on gère le cas où le dossier existe déjà)
for file_name in os.listdir(source_audio):
    if file_name.endswith('.mp3'):
        full_file_name = os.path.join(source_audio, file_name)
        if os.path.isfile(full_file_name):
            shutil.copy(full_file_name, target_audio)

print("Transfert vers le Drive terminé !")

Copie des fichiers de /content/giantsteps-tempo-dataset/audio vers /content/drive/MyDrive/PJT IA Musicale/Datasets/GiantSteps...
Transfert vers le Drive terminé ! Vous pouvez vérifier votre dossier.


On convertit désormais chacun des fichiers audio dans une version .wav

In [ ]:
from pydub import AudioSegment
import os
import glob

target_audio = '/content/drive/MyDrive/PJT IA Musicale/Datasets/GiantSteps'
working_dir = '/content/drive/MyDrive/PJT IA Musicale'


data_dir=target_audio

list_of_files = glob.glob(os.path.join(data_dir, "*.mp3"))

i=0 # Application sur un des fichiers
print(list_of_files)

def mp3_to_wav(input_file, output_file):
    # Charger le fichier MP3
    output_dir = os.path.dirname(output_file)
    if not os.path.exists(output_dir):
      os.makedirs(output_dir) # pour créer le fichier s'il n'existe pas
    audio = AudioSegment.from_mp3(input_file)
    # Exporter en WAV
    audio.export(output_file, format="wav")
    print(f"Conversion terminée : {output_file}")


if list_of_files: #on procède seulement si fichier mp3 sont existants
    for i in range(len(list_of_files)):
        input_mp3 = list_of_files[i]   # Remplacez par votre fichier mp3
        # Corrected: Construct the output WAV path using os.path.join and os.path.splitext
        file_name_without_ext = os.path.splitext(os.path.basename(input_mp3))[0]
        output_wav = os.path.join(working_dir, "Datasets", "Giantsteps_converted_wav", file_name_without_ext + ".wav")
        mp3_to_wav(input_mp3,output_wav)
        print(i,"/",len(list_of_files))
    print("Toutes les conversions sont terminées !")
else:
    print("Aucun fichier .mp3 trouvé dans le répertoire spécifié.")

e/Datasets/GiantSteps/4162511.LOFI.mp3', '/content/drive/MyDrive/PJT IA Musicale/Datasets/GiantSteps/4162259.LOFI.mp3']
Conversion terminée : /content/drive/MyDrive/PJT IA Musicale/Datasets/Giantsteps_converted_wav/3368053.LOFI.wav
0 / 664
Conversion terminée : /content/drive/MyDrive/PJT IA Musicale/Datasets/Giantsteps_converted_wav/1905592.LOFI.wav
1 / 664
Conversion terminée : /content/drive/MyDrive/PJT IA Musicale/Datasets/Giantsteps_converted_wav/4960424.LOFI.wav
2 / 664
Conversion terminée : /content/drive/MyDrive/PJT IA Musicale/Datasets/Giantsteps_converted_wav/2088281.LOFI.wav
3 / 664


# Vérification que les fichiers ont été convertis au bonne endroit,  ainsi que le nombre de dossier téléchargé


In [ ]:
working_dir = '/content/drive/MyDrive/PJT IA Musicale/Datasets/Giantsteps_converted_wav'

list_of_files = glob.glob(os.path.join(working_dir, "*.wav"))

i=0 # Application sur un des fichiers
print(len(list_of_files))
print(list_of_files[i])

668
/content/drive/MyDrive/PJT IA Musicale/Datasets/Giantsteps_converted_wav/128.0_3368053.LOFI.wav


# **2. Fonctions de Pré-traitement (Spectrogrammes pour BPM)**

2.1 associer les fichiers audio  à leur bpm


Les bpm associés aux musiques sont stockeés suyr la page gihub
https://github.com/GiantSteps/giantsteps-tempo-dataset?tab=readme-ov-file
dans le dossier annotations -> Tempo

In [ ]:
import os
import shutil

# 1. Source (GitHub temporaire)
source_annotations = '/content/giantsteps-tempo-dataset/annotations/tempo'

# 2. Destination
# On crée un dossier dédié à côté de vos fichiers audio
target_annotations = '/content/drive/MyDrive/PJT IA Musicale/Datasets/GiantSteps_annotations/tempo'

# Vérification si le clonage est toujours actif, sinon on le refait vite fait
if not os.path.exists(source_annotations):
    print("Le dossier source n'existe plus, on re-clone le repo...")
    !git clone https://github.com/GiantSteps/giantsteps-tempo-dataset.git /content/giantsteps-tempo-dataset

# 3. Copie
if not os.path.exists(target_annotations):
    print(f"Copie des annotations vers {target_annotations}...")
    shutil.copytree(source_annotations, target_annotations)
    print("✅ Transfert terminé ! Vos annotations sont sécurisées sur le Drive.")
else:
    print("ℹ️ Le dossier annotations existe déjà sur le Drive. Pas besoin de copier.")

ℹ️ Le dossier annotations existe déjà sur le Drive. Pas besoin de copier.


 Associer les les BPM trouvés au bon fichier audio





In [ ]:
import os
import glob
import numpy as np
from tqdm.auto import tqdm # Pour avoir la barre de progression

#  1. CONFIGURATION DES CHEMINS
wav_dir = '/content/drive/MyDrive/PJT IA Musicale/Datasets/Giantsteps_converted_wav'

bpm_dir = '/content/drive/MyDrive/PJT IA Musicale/Datasets/GiantSteps_annotations/tempo'


# 2. fonction de lecture style bloc note , seule façon d elire les bpm dans le fichiers
def lire_bpm_dans_fichier(bpm_path):
    try:
        with open(bpm_path, 'r') as f:
            contenu = f.read().strip() #on lit f et on enlève si il y'en a les espaces et retour à la ligne
            return float(contenu)
    except Exception as e:
        return None

#3. association
print("Début de l'association Audio <-> BPM...")

X_paths = [] # Chemins audio
Y_labels = [] # BPMs

# On liste tous les fichiers wav
wav_files = glob.glob(os.path.join(wav_dir, "*.wav"))

compteur_succes = 0
compteur_echec = 0

for wav_path in tqdm(wav_files):
    # Etape 1 : Extraire le nom du fichier(ex: '/12345.wav' -> '12345')
    nom_fichier = os.path.basename(wav_path)
    id_fichier = os.path.splitext(nom_fichier)[0] #pour enlever le .wav

    # Etape 2 : Construire le chemin théorique du fichier BPM
    # On cherche '12345.bpm' dans le dossier tempo
    chemin_bpm_theorique = os.path.join(bpm_dir, id_fichier + ".bpm")

    # Etape 3 : Vérifier si le fichier existe et le lire
    if os.path.exists(chemin_bpm_theorique):
        bpm_valeur = lire_bpm_dans_fichier(chemin_bpm_theorique)

        if bpm_valeur is not None:
            # C'est bon, on a trouvé la paire
            X_paths.append(wav_path) #dans ce cas là on dirige les dossieres vers X_paths et Y_paths
            Y_labels.append(bpm_valeur)
            compteur_succes += 1
        else:
            compteur_echec += 1
    else:
        # Parfois les fichiers s'appellent id.lofi.bpm, essayons ça si le premier échoue
        chemin_bpm_alt = os.path.join(bpm_dir, id_fichier + ".lofi.bpm")
        if os.path.exists(chemin_bpm_alt):
             bpm_valeur = lire_bpm_dans_fichier(chemin_bpm_alt)
             if bpm_valeur is not None:
                X_paths.append(wav_path)
                Y_labels.append(bpm_valeur)
                compteur_succes += 1
        else:
            compteur_echec += 1

# Conversion en tableaux Numpy pour le réseau de neuronnes
X_paths = np.array(X_paths)
Y_labels = np.array(Y_labels)

print("\n" + "="*30)
print(f"TERMINÉ.")
print(f"Paires réussies : {compteur_succes}")
print(f"Echecs (pas de bpm trouvé) : {compteur_echec}")
print("="*30)

if len(X_paths) > 0:
    print(f"Test 1 : {os.path.basename(X_paths[0])} correspond à {Y_labels[0]} BPM")


TERMINÉ.
Paires réussies : 0
Echecs (pas de bpm trouvé) : 664


On a bien pour le fichier 3368053 un bpm de 128 sur github

Maintenant l'association faites, on va formatter les données d'entrée


In [ ]:
import os
from tqdm.auto import tqdm

# On s'assure que X_paths et Y_labels existent bien depuis l'étape précédente
if 'X_paths' not in locals() or 'Y_labels' not in locals():
    print("❌ Erreur : Les variables X_paths et Y_labels ne sont pas en mémoire.")
    print("Veuillez relancer le bloc de code précédent qui faisait l'association.")
else:
    print(f"Démarrage du renommage pour {len(X_paths)} fichiers...")

    compteur_rename = 0
    compteur_deja_fait = 0

    # On utilise zip pour parcourir le fichier et son BPM en même temps
    for old_path, bpm in tqdm(zip(X_paths, Y_labels), total=len(X_paths)):

        # 1. Décomposition du chemin
        directory = os.path.dirname(old_path)
        filename = os.path.basename(old_path)

        # 2. Formatage du BPM (ex: 120.0)
        # On garde 1 décimale pour la précision, mais pas trop pour la lisibilité
        bpm_prefix = f"{bpm:.1f}"

        # 3. VÉRIFICATION DE SÉCURITÉ (Idempotence)
        # Si le fichier commence déjà par le BPM (ex: "120.0_..."), on ne touche à rien
        if filename.startswith(bpm_prefix + "_"):
            compteur_deja_fait += 1
            continue

        # 4. Construction du nouveau nom
        new_filename = f"{bpm_prefix}_{filename}"
        new_full_path = os.path.join(directory, new_filename)

        # 5. Renommage effectif sur le disque (Drive)
        try:
            os.rename(old_path, new_full_path)
            compteur_rename += 1
        except OSError as e:
            print(f"⚠️ Erreur système sur {filename} : {e}")

    print("\n" + "="*30)
    print("RAPPORT D'OPÉRATION")
    print(f"✅ Fichiers renommés avec succès : {compteur_rename}")
    print(f"ℹ️ Fichiers ignorés (déjà renommés) : {compteur_deja_fait}")
    print("="*30)

    # Vérification
    print("Nouveau nom exemple :", new_filename)

In [ ]:
import os
import glob
import numpy as np
import librosa
from tqdm.auto import tqdm
SAMPLE_RATE = 22050

# Durée du morceau qu'on va analyser (en secondes).
# Contrairement aux chiffres prononcé par les speakers durant le PJT1A (les fichiers avaient une durée de 1se), pour le BPM il faut écouter plus longtemps.
# On coupe tous les morceaux à  secondes pour avoir une entrée fixe pour le réseau de neurone .
DURATION = 15
SAMPLES_PER_TRACK = SAMPLE_RATE * DURATION

#1.directions des fichiers

working_dir = '/content/drive/MyDrive/PJT IA Musicale/Datasets/Giantsteps_converted_wav'
list_of_files = glob.glob(os.path.join(working_dir, "*.wav"))

#2. fonctions

def get_bpm_from_filename(filename):
    try:
        # On découpe le nom par les '_' et on prend le premier morceau
        bpm_str = filename.split('_')[0]
        return float(bpm_str)
    except Exception as e:
        print(f"Attention!! Erreur format nom : {filename} ({e})")
        return None

def preprocess_audio(file_path):
    try:
        # 1.on charge l'audio
        signal, sr = librosa.load(file_path, sr=SAMPLE_RATE, duration=DURATION)

        # 2.On formate les signaux d'entrée pour qu'ils aient tous la même taille
        if len(signal) < SAMPLES_PER_TRACK:
            padding = SAMPLES_PER_TRACK - len(signal) # Si trop court, on complète avec des zéros
            signal = np.pad(signal, (0, padding), 'constant')
        else:
            signal = signal[:SAMPLES_PER_TRACK] # Si trop long, on coupe

        # 3.Conversion en melspec
        mel_spec = librosa.feature.melspectrogram(y=signal,sr=sr,n_fft=2048,hop_length=512,n_mels=128)

        # 4.Passage en dB
        mel_spec_db = librosa.power_to_db(mel_spec, ref=np.max)

        # 5. Normalisation entre 0 et 1
        # pour la convergence du gradient descent
        norm_spec = (mel_spec_db - mel_spec_db.min()) / (mel_spec_db.max() - mel_spec_db.min())

        # Ajout dimension canal pour le réseau de neurone
        return norm_spec[..., np.newaxis]

    except Exception as e:
        print(f"❌ Erreur sur {file_path}: {e}")
        return None

# 3. execution des fonctions
print(f"Démarrage du traitement sur {len(list_of_files)} fichiers...")

X = []
Y = []

if not list_of_files:
    print("Erreur: Dossier vide ou chemin incorrect.")
else:
    for file_path in tqdm(list_of_files):
        # 1. Extraction Label
        bpm = get_bpm_from_filename(os.path.basename(file_path))

        # 2. Extraction Features (si Label ok)
        if bpm is not None:
            spec = preprocess_audio(file_path)

            if spec is not None:
                X.append(spec)
                Y.append(bpm)

    # Conversion finale
    X = np.array(X)
    Y = np.array(Y)

    print("\n" + "="*30)
    print("PIPELINE TERMINÉ")
    if len(X) > 0:
        print(f"Taille du dataset X : {X.shape}")
        print(f"Taille des labels Y : {Y.shape}")
        print(f"Exemple Y : {Y[:3]}")
    else:
        print("Aucune donnée générée.")

Démarrage du traitement sur 668 fichiers...


In [ ]:
import os
import numpy as np

# On crée un dossier dédié aux "tenseurs" prêts à l'emploi
save_dir = '/content/drive/MyDrive/PJT IA Musicale/Datasets/Processed_Data'

if not os.path.exists(save_dir):
    os.makedirs(save_dir)
    print(f"Dossier créé : {save_dir}")

# --- SAUVEGARDE ---
print("Sauvegarde en cours sur le Drive ")

# On utilise le format natif de Numpy (.npy) qui est ultra-rapide et binaire
path_X = os.path.join(save_dir, 'X_giantsteps.npy')
path_Y = os.path.join(save_dir, 'Y_giantsteps.npy')

np.save(path_X, X)
np.save(path_Y, Y)

# --- VÉRIFICATION ---
# On vérifie la taille des fichiers pour être sûr que ce n'est pas vide
size_X = os.path.getsize(path_X) / (1024 * 1024) # En Mo
size_Y = os.path.getsize(path_Y) / (1024 * 1024) # En Mo

print(f"\n✅ SAUVEGARDE RÉUSSIE !")
print(f"X (Images) sauvegardé ici : {path_X} ({size_X:.2f} Mo)")
print(f"Y (Labels) sauvegardé ici : {path_Y} ({size_Y:.2f} Mo)")

Sauvegarde en cours sur le Drive 


Pour appeler directement X et Y


In [ ]:

import numpy as np

print("Chargement des données depuis le Drive")

# Chemins (à adapter si besoin)
path_X = '/content/drive/MyDrive/PJT IA Musicale/Datasets/Processed_Data/X_giantsteps.npy'
path_Y = '/content/drive/MyDrive/PJT IA Musicale/Datasets/Processed_Data/Y_giantsteps.npy'

# Chargement instantané
X = np.load(path_X)
Y = np.load(path_Y)

print(f"Données chargées ! X: {X.shape}, Y: {Y.shape}")
# Vous pouvez directemenent lancer model.fit(X, Y, ...)

Chargement des données depuis le Drive


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import os

# 1. Chargement des données
if 'Y' not in locals():
    print("Chargement de Y depuis le fichier de sauvegarde...")
    try:
        path_Y = '/content/drive/MyDrive/PJT IA Musicale/Datasets/Processed_Data/Y_giantsteps.npy'
        Y = np.load(path_Y)
    except FileNotFoundError:
        print(f"⚠️ Erreur : Impossible de trouver le fichier {path_Y}")
        # On arrête là si pas de données
        raise

#2 caclul de la distribution des données
mean_bpm = np.mean(Y)
median_bpm = np.median(Y)
std_bpm = np.std(Y)
min_bpm = np.min(Y)
max_bpm = np.max(Y)

print(f"--- ANALYSE STATISTIQUE ---")
print(f"Nombre d'échantillons : {len(Y)}")
print(f"BPM Moyen      : {mean_bpm:.2f}")
print(f"BPM Médian     : {median_bpm:.2f}")
print(f"Écart-type     : {std_bpm:.2f} (Dispersion des données)")
print(f"Plage          : [{min_bpm:.1f} - {max_bpm:.1f}] BPM")
print("---------------------------")

# 3. Visualisation
plt.figure(figsize=(12, 6))

# On crée l'histogramme avec des "bins" (barres) tous les 5 BPM pour la lisibilité
plt.hist(Y, bins=range(int(min_bpm), int(max_bpm) + 5, 5),
         color='#3498db', edgecolor='black', alpha=0.7)

# Ajout des lignes verticales pour la moyenne et la médiane
plt.axvline(mean_bpm, color='red', linestyle='dashed', linewidth=2, label=f'Moyenne ({mean_bpm:.1f})')
plt.axvline(median_bpm, color='green', linestyle='dashed', linewidth=2, label=f'Médiane ({median_bpm:.1f})')
plt.axvline(mean_bpm+std_bpm, color='blue', linestyle='dashed', linewidth=2, label=f'moyenne+éecarttype ({mean_bpm+std_bpm:.1f})')
plt.axvline(mean_bpm-std_bpm, color='blue', linestyle='dashed', linewidth=2, label=f'moyenne-éecarttyp ({mean_bpm-std_bpm:.1f})')

plt.title('Distribution des BPM dans le Dataset GiantSteps', fontsize=15)
plt.xlabel('BPM (Battements par minute)', fontsize=12)
plt.ylabel('Nombre de morceaux', fontsize=12)
plt.legend()
plt.grid(axis='y', alpha=0.5)

plt.show()

--- ANALYSE STATISTIQUE ---
Nombre d'échantillons : 664
BPM Moyen      : 136.66
BPM Médian     : 140.00
Écart-type     : 28.33 (Dispersion des données)
Plage          : [53.0 - 200.0] BPM
---------------------------


In [ ]:
plt.figure(figsize=(12, 6))
Y=[i for i in range(60,200)]
# On crée l'histogramme avec des "bins" (barres) tous les 5 BPM pour la lisibilité
plt.hist(Y, bins=range(int(60), int(200) + 5, 5),
         color='#3498db', edgecolor='black', alpha=0.7)


plt.title('Distribution ideal des BPM dans le Dataset GiantSteps', fontsize=15)
plt.xlabel('BPM (Battements par minute)', fontsize=12)
plt.ylabel('Nombre de morceaux', fontsize=12)
plt.legend()
plt.grid(axis='y', alpha=0.5)

plt.show()

/tmp/ipython-input-1830594929.py:11: UserWarning: No artists with labels found to put in legend.  Note that artists whose label start with an underscore are ignored when legend() is called with no argument.
  plt.legend()


On peut déja émettre l'hypothèse que le nombre de fichier audio n'est toujours pas suffisant, étant donne que le réseau va chercher à minimser son erreur en renvoyant des valeurs proche de 140 il a peu de chance de se tromper notr réseau va donc ignorer les valeurs extrmes de BPM



modèle 1: non concluant


In [ ]:
import numpy as np
import tensorflow as tf
from sklearn.model_selection import train_test_split
from tensorflow.keras import layers, models, optimizers
X_train, X_test, y_train, y_test = train_test_split(X, Y, test_size=0.2, random_state=42)

print(f"Entraînement sur {len(X_train)} échantillons")
print(f"Validation sur {len(X_test)} échantillons")


Entraînement sur 531 échantillons
Validation sur 133 échantillons


ne plus lancer modèle moins performant

In [ ]:
import numpy as np
import tensorflow as tf
from sklearn.model_selection import train_test_split
from tensorflow.keras import layers, models, optimizers

# 1. Chargement des données si nécessaire
# X = np.load('/content/drive/MyDrive/PJT IA Musicale/Datasets/Processed_Data/X_giantsteps.npy')
# Y = np.load('/content/drive/MyDrive/PJT IA Musicale/Datasets/Processed_Data/Y_giantsteps.npy')

print(f"Entraînement sur {len(X_train)} échantillons")
print(f"Validation sur {len(X_test)} échantillons")

# 2. Définition de l'Architecture

def build_bpm_model(input_shape):
    model = models.Sequential()

    #  Bloc1
    model.add(layers.Conv2D(16, (3, 3), activation='relu', padding='same', input_shape=input_shape))
    model.add(layers.MaxPooling2D((2, 2))) # Réduit la taille par 2 (garde l'essentiel)
    model.add(layers.BatchNormalization())

    #  Bloc2
    model.add(layers.Conv2D(32, (3, 3), activation='relu', padding='same'))
    model.add(layers.MaxPooling2D((2, 2)))
    model.add(layers.BatchNormalization())

    #  Bloc3
    model.add(layers.Conv2D(64, (3, 3), activation='relu', padding='same'))
    model.add(layers.MaxPooling2D((2, 2)))
    model.add(layers.BatchNormalization())

   #  Bloc4
    model.add(layers.Conv2D(128, (3, 3), activation='relu', padding='same'))
    model.add(layers.MaxPooling2D((2, 2)))
    model.add(layers.BatchNormalization())

    #applatissement
    model.add(layers.Flatten()) # Transforme l'image 3D en vecteur 1D

    model.add(layers.Dense(64, activation='relu') )
    model.add(layers.Dropout(0.3)) # Éteint 30% des neurones au hasard pour éviter le par coeur (Overfitting)

    # Couche de sortie
    # Pas d'activation (linéaire) car on veut un nombre réel (ex: 124.5), pas une probabilité 0-1
    model.add(layers.Dense(1))
    return model

# Récupération de la forme d'une image (ex: 128 x 431 x 1)
input_shape = X_train[0].shape
model = build_bpm_model(input_shape)

# 3. Compilation
# Loss : MAE (Mean Absolute Error).: "Le réseau se trompe de X BPM en moyenne"
model.compile(optimizer=optimizers.Adam(learning_rate=0.001),loss='mae',metrics=['mse'])

model.summary()

/usr/local/lib/python3.12/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


modèle 2: en cours

In [ ]:
from tensorflow.keras import layers, models, optimizers

def build_lightweight_model(input_shape):
    model = models.Sequential()

    # Bloc 1
    model.add(layers.Conv2D(16, (3, 3), activation='relu', padding='same', input_shape=input_shape))
    model.add(layers.MaxPooling2D((2, 2)))
    model.add(layers.BatchNormalization())

   # Bloc 2
    model.add(layers.Conv2D(32, (3, 3), activation='relu', padding='same'))
    model.add(layers.MaxPooling2D((2, 2)))
    model.add(layers.BatchNormalization())

    # Bloc 3
    model.add(layers.Conv2D(64, (3, 3), activation='relu', padding='same'))
    model.add(layers.MaxPooling2D((2, 2)))
    model.add(layers.BatchNormalization())

   # Bloc 4
    model.add(layers.Conv2D(128, (3, 3), activation='relu', padding='same'))
    model.add(layers.MaxPooling2D((2, 2)))
    model.add(layers.BatchNormalization())


    # Au lieu de Flatten(), on prend la moyenne de chaque canal.
    model.add(layers.GlobalAveragePooling2D())# on a testé la fonction globaleveragepooliing 2D

    # On garde une petite couche dense pour la flexibilité
    model.add(layers.Dense(64, activation='relu'))
    model.add(layers.Dropout(0.5)) # Dropout plus agressif (50%)

    # Sortie
    model.add(layers.Dense(1))

    return model

# Re-construction et compilation
input_shape = X_train[0].shape
model = build_lightweight_model(input_shape)

# On réduit un peu le Learning Rate pour stabiliser (0.001 -> 0.0005)
model.compile(optimizer=optimizers.Adam(learning_rate=0.0005),loss='mae',metrics=['mse'])

model.summary()

/usr/local/lib/python3.12/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [ ]:
import matplotlib.pyplot as plt

# 1. Configuration de l'entraînement
# Batch size 32 est un standard (compromis vitesse/stabilité)
# Epochs 50 : On lui laisse le temps, on arrêtera avant si besoin
history = model.fit(
    X_train, y_train,
    epochs=50,
    batch_size=32,
    validation_data=(X_test, y_test),
    verbose=1 # Affiche la barre de progression
)

# 2. Visualisation des courbes (Tableau de bord de l'ingénieur)
def plot_history(history):
    # MAE is the loss in this model setup
    mae = history.history['loss']
    val_mae = history.history['val_loss']
    # MSE is the metric in this model setup
    mse = history.history['mse']
    val_mse = history.history['val_mse']
    epochs = range(1, len(mae) + 1)

    plt.figure(figsize=(15, 5))

    # Graphe 1 : L'erreur absolue (MAE) - C'est ce qui nous intéresse (l'erreur en BPM)
    plt.subplot(1, 2, 1)
    plt.plot(epochs, mae, 'b', label='Entraînement MAE')
    plt.plot(epochs, val_mae, 'r', label='Validation MAE')
    plt.title('Erreur Absolue Moyenne (BPM)')
    plt.xlabel('Époques')
    plt.ylabel('Erreur (BPM)')
    plt.legend()
    plt.grid(True)

    # Graphe 2 : La Loss (MSE) - C'est ce que le math optimise
    plt.subplot(1, 2, 2)
    plt.plot(epochs, mse, 'b', label='Entraînement MSE') # Changed to mse
    plt.plot(epochs, val_mse, 'r', label='Validation MSE') # Changed to val_mse
    plt.title('Fonction de Coût (MSE)') # Changed title
    plt.xlabel('Époques')
    plt.legend()
    plt.grid(True)

    plt.show()

# 3. Affichage
plot_history(history)


Epoch 1/50
 5/17 ━━━━━━━━━━━━━━━━━━━━ 2s 184ms/step - loss: 130.1655 - mse: 17893.3594

Modèle avec callback

In [ ]:
from tensorflow.keras.callbacks import ReduceLROnPlateau, ModelCheckpoint, EarlyStopping

# 1. Définition des Callbacks (Les gardes-fous)

# A. Réduire le Learning Rate si ça stagne
reduce_lr = ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.5,        # On divise le LR par 2
    patience=10,        # Si pas d'amélioration pendant 10 époques
    min_lr=0.00001,    # Ne pas descendre en dessous de ça
    verbose=1          # Affiche un message quand il réduit
)

# B. Sauvegarder UNIQUEMENT le meilleur modèle
checkpoint = ModelCheckpoint(
    filepath='/content/drive/MyDrive/PJT IA Musicale/Datasets/Processed_Data/best_model.keras',
    monitor='val_loss',
    save_best_only=True, # Important ! Garde l'époque 47, jette la 50
    verbose=1
)

# C. Arrêt anticipé (si vraiment ça ne s'améliore plus du tout)
early_stop = EarlyStopping(
    monitor='val_loss',
    patience=50,       # On attend 30 époques avant d'abandonner
    restore_best_weights=True
)

# 2. Relancer l'entraînement (On peut tenter 100 époques maintenant)
print("Relance de l'entraînement avec Callbacks...")

history_v2 = model.fit(
    X_train, y_train,
    epochs=1000,            # On augmente car le LR va baisser
    batch_size=32,
    validation_data=(X_test, y_test),
    callbacks=[reduce_lr, checkpoint, early_stop], # On branche les outils
    verbose=1
)

17/17 ━━━━━━━━━━━━━━━━━━━━ 3s 202ms/step - loss: 27.4882 - mse: 1318.8578 - val_loss: 24.4360 - val_mse: 1209.5887 - learning_rate: 3.1250e-05
Epoch 67/1000
17/17 ━━━━━━━━━━━━━━━━━━━━ 0s 185ms/step - loss: 25.8172 - mse: 1238.7867
Epoch 67: ReduceLROnPlateau reducing learning rate to 1.5625000742147677e-05.

Epoch 67: val_loss did not improve from 20.32990
17/17 ━━━━━━━━━━━━━━━━━━━━ 4s 207ms/step - loss: 25.7882 - mse: 1236.0892 - val_loss: 23.5942 - val_mse: 1174.0310 - learning_rate: 3.1250e-05


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from tensorflow.keras.models import load_model

def plot_history(history):
    # MAE is the loss in this model setup
    mae = history.history['loss']
    val_mae = history.history['val_loss']
    # MSE is the metric in this model setup
    mse = history.history['mse']
    val_mse = history.history['val_mse']
    epochs = range(1, len(mae) + 1)

    plt.figure(figsize=(15, 5))

    # Graphe 1 : L'erreur absolue (MAE) - C'est ce qui nous intéresse (l'erreur en BPM)
    plt.subplot(1, 2, 1)
    plt.plot(epochs, mae, 'b', label='Entraînement MAE')
    plt.plot(epochs, val_mae, 'r', label='Validation MAE')
    plt.title('Erreur Absolue Moyenne (BPM)')
    plt.xlabel('Époques')
    plt.ylabel('Erreur (BPM)')
    plt.legend()
    plt.grid(True)

    # Graphe 2 : La Loss (MSE) - C'est ce que le math optimise
    plt.subplot(1, 2, 2)
    plt.plot(epochs, mse, 'b', label='Entraînement MSE') # Changed to mse
    plt.plot(epochs, val_mse, 'r', label='Validation MSE') # Changed to val_mse
    plt.title('Fonction de Coût (MSE)') # Changed title
    plt.xlabel('Époques')
    plt.legend()
    plt.grid(True)
# 1. Chargement du Champion (Epoch 19)
print("Chargement du meilleur modèle...")
best_model_path = '/content/drive/MyDrive/PJT IA Musicale/Datasets/Processed_Data/best_model.keras'
model = load_model(best_model_path)

# 2. Prédictions sur le jeu de test (que le modèle n'a jamais vu)
print("Génération des prédictions sur X_test...")
predictions = model.predict(X_test)

# Aplatir les résultats (passer de [[120], [130]] à [120, 130])
y_pred = predictions.flatten()


def accuracy_tolerance(y_true, y_pred, tolerance=0.05): # ici on considère la tolérance à 5%
    errors = np.abs(y_true - y_pred)
    # On regarde si l'erreur est inférieure à 5% de la valeur réelle
    is_correct = errors <= (tolerance * y_true)
    return np.mean(is_correct) * 100

acc_strict = accuracy_tolerance(y_test, y_pred)

print(f"\n--- RÉSULTATS DÉTAILLÉS ---")
print(f"Précision Stricte (10% tolérance) : {acc_strict:.2f}%")
print(f"MAE Finale (Moyenne des erreurs) : {np.mean(np.abs(y_test - y_pred)):.2f} BPM")


# Si les points forment une ligne diagonale parfaite, c'est gagné ( un peu comme la matrice de confusion de classification )
# Si on voit des lignes parallèles (au dessus ou en dessous), c'est l'erreur d'octave.
plt.figure(figsize=(10, 10))
plt.scatter(y_test, y_pred, alpha=0.6, c='blue', edgecolors='k', s=50)

# Ligne parfaite (x=y)
plt.plot([50, 200], [50, 200], 'r--', linewidth=2, label="Parfait (x=y)")

# Lignes d'erreur d'octave (x=2y et x=0.5y)
plt.plot([50, 200], [100, 400], 'g--', alpha=0.5, label="Double Tempo (Erreur Octave)")
plt.plot([50, 200], [25, 100], 'g--', alpha=0.5, label="Demi Tempo (Erreur Octave)")

plt.title("Réalité vs Prédiction IA", fontsize=16)
plt.xlabel("Vrai BPM (Annotations)", fontsize=14)
plt.ylabel("BPM Prédit par l'IA", fontsize=14)
plt.xlim(50, 200)
plt.ylim(50, 200)
plt.legend()
plt.grid(True)
plt.show()
plot_history(history_v2)

Chargement du meilleur modèle...
Génération des prédictions sur X_test...
5/5 ━━━━━━━━━━━━━━━━━━━━ 2s 247ms/step

--- RÉSULTATS DÉTAILLÉS ---
Précision Stricte (10% tolérance) : 44.36%
MAE Finale (Moyenne des erreurs) : 20.46 BPM
